# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source follows the [Croissant schema](https://mlcommons.org/croissant/) and is provided via a schema URL. This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes fields such as protein accession, description, coverage percentage, peptide counts, molecular weight (MW), and calculated parameters such as pI and normalized abundances across different samples.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access key metadata attributes
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published on: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(dataset.metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, their `@id`s, fields, and columns. This will help in selecting data for analysis. All entity references use their `@id` according to the Croissant schema.

In [ ]:
# List all record sets in the dataset
record_sets = [rs['@id'] for rs in dataset.metadata.recordSets]
print('Available record sets and their @id:')
for rs in dataset.metadata.recordSets:
    print(f"- {rs.get('@id')}: {rs.get('name', '<no name>')}")
    # List fields for each record set
    fields = rs.get('fields', [])
    for fld in fields:
        print(f"    * Field @id: {fld.get('@id', '<no id>')} - {fld.get('name', '<no name>')}")
    # List columns
    columns = rs.get('columns', [])
    for col in columns:
        print(f"    * Column @id: {col.get('@id', '<no id>')} - {col.get('name', '<no name>')}")

**Example Record Preview:**

Let's preview a few records from the main record set using its `@id`:

In [ ]:
# Manually specify the main record set @id for this dataset
# (as recordSets may be empty in root schema, we will use the data table @id seen in download)
# In FAIR², main data is typically under '@id': 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
main_recordset_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'

print(f"First 2 records from record set {main_recordset_id}:\n")
for i, rec in enumerate(dataset.records(record_set=main_recordset_id)):
    print(rec)
    if i >= 1:
        break

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. We reference record sets and fields using their exact `@id`s per the Croissant schema.

In [ ]:
# List of available record set @ids (if your dataset has more, you can add here)
record_set_ids = [main_recordset_id]
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    # Load into DataFrame
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Fields (columns) in {rs_id}:")
    print(df.columns.tolist())
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Let's process the main data table for further analysis.

- We'll choose a numeric field (such as peptide counts or coverage percentage) for filtering and normalization.
- We'll use the field `@id` directly for referencing.
- Example: Filter proteins with coverage (%) > 10, normalize this field, and group by 'accession' if present.

In [ ]:
# Select a numeric field by @id (e.g., 'Protein Coverage (%)')
df = dataframes[main_recordset_id]
# Let's try to pick the field by the column name available
numeric_candidates = [col for col in df.columns if '%' in col or col.lower().strip().startswith('coverage') or 'peptide' in col.lower() or 'abundance' in col.lower()]
print(f"Detected numeric fields (candidates): {numeric_candidates}")
if numeric_candidates:
    numeric_field = numeric_candidates[0]  # For example: 'Protein Coverage (%)'
else:
    numeric_field = df.select_dtypes('number').columns[0]  # fallback
print(f"Using numeric column: {numeric_field}")

# Clean/convert the numeric column (in case it's string with % or commas)
def to_float(x):
    try:
        if isinstance(x, str) and '%' in x:
            return float(x.replace('%', '').strip())
        return float(x)
    except:
        return pd.NA

df[numeric_field] = df[numeric_field].apply(to_float)
threshold = 10.0

filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Number of records with {numeric_field} > {threshold}: {len(filtered_df)}")
print(filtered_df[[numeric_field]].head(3))

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} (z-score):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))

# Try grouping by accession field or similar identifier
group_candidates = [col for col in df.columns if 'accession' in col.lower() or 'id' in col.lower()]
group_field = group_candidates[0] if group_candidates else None
if group_field:
    grouped_df = (
        filtered_df.groupby(group_field)
        .agg({numeric_field: 'mean', f"{numeric_field}_normalized": 'mean'})
        .reset_index()
    )
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head(3))

## 5. Visualization

Let's visualize the distribution of the chosen numeric field after filtering and normalization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f'Distribution of {numeric_field} (> {threshold})')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True, color='salmon')
    plt.title(f'Normalized ({numeric_field}) Distribution (z-score)')
    plt.xlabel(f'{numeric_field} (normalized)')
    plt.ylabel('Frequency')
    plt.show()

## 6. Conclusion

- We have loaded and previewed protein-level data from the FAIR² dataset defined by the Croissant schema.
- We filtered and normalized key numeric fields, with candidate grouping by accession.
- The data appears suitable for further mass spectrometry EDA, such as comparing protein modifications or abundances across experiments.

For more advanced processing, refer to the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/latest/user/usage.html) and the full FAIR² dataset schema.